In [ ]:
#ResNet152
import numpy as np

# -----------------------------------------------
# Cohen's kappa
# -----------------------------------------------
def cohens_kappa(cm):
    N = np.sum(cm)
    Po = np.trace(cm) / N
    row = np.sum(cm, axis=1)
    col = np.sum(cm, axis=0)
    Pe = np.sum(row * col) / (N * N)
    kappa = (Po - Pe) / (1 - Pe)
    var_k = (Po * (1 - Po)) / (N * (1 - Pe)**2)
    return kappa, var_k

# -----------------------------------------------
# Bootstrap CI on kappa difference
# -----------------------------------------------
def bootstrap_kappa_diff(cm1, cm2, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    N = int(cm1.sum())
    n_classes = cm1.shape[0]

    # Expand CM into sample-level (true_label, pred_label) pairs
    def cm_to_samples(cm):
        samples = []
        for i in range(n_classes):
            for j in range(n_classes):
                samples.extend([(i, j)] * cm[i, j])
        return np.array(samples)   # shape (N, 2)

    s1 = cm_to_samples(cm1)   # model1 samples
    s2 = cm_to_samples(cm2)   # model2 samples

    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, N, size=N)   # resample WITH replacement

        # Rebuild CMs from bootstrap indices
        boot_cm1 = np.zeros((n_classes, n_classes), dtype=int)
        boot_cm2 = np.zeros((n_classes, n_classes), dtype=int)
        for true_l, pred_l in s1[idx]:
            boot_cm1[true_l, pred_l] += 1
        for true_l, pred_l in s2[idx]:
            boot_cm2[true_l, pred_l] += 1

        k1, _ = cohens_kappa(boot_cm1)
        k2, _ = cohens_kappa(boot_cm2)
        diffs.append(k2 - k1)   # proposed - base

    diffs = np.array(diffs)

    # Observed difference
    k1_obs, _ = cohens_kappa(cm1)
    k2_obs, _ = cohens_kappa(cm2)
    obs_diff = k2_obs - k1_obs

    # Percentile CI
    ci_low  = np.percentile(diffs, 2.5)
    ci_high = np.percentile(diffs, 97.5)

    # Continuity-corrected two-sided bootstrap p-value.
    # Adding +1 to numerator and denominator avoids an exact p=0 when all
    # resamples fall on one side — a standard Monte Carlo correction.
    count_le = np.sum(diffs <= 0)
    count_ge = np.sum(diffs >= 0)
    p_val = 2 * min(
        (count_le + 1) / (n_boot + 1),
        (count_ge + 1) / (n_boot + 1)
    )

    return obs_diff, ci_low, ci_high, p_val, diffs, k1_obs, k2_obs, count_le, count_ge

# -----------------------------------------------
# Helper: format p-value honestly
# -----------------------------------------------
def format_p(p_val, n_boot):
    """
    When all bootstrap resamples fall on one side, the continuity-corrected
    p equals 2/(n_boot+1).  Report this as a bound rather than a spurious
    exact value.
    """
    resolution = 2 / (n_boot + 1)
    if p_val <= resolution:
        return f"< {resolution:.4f}  (below bootstrap resolution)"
    elif p_val < 0.001:
        return f"{p_val:.2e}"
    else:
        return f"{p_val:.4f}"

# -----------------------------------------------
# Pretty-print results for one experiment
# -----------------------------------------------
def run_analysis(label, cm_model1, cm_model2, n_boot=1000, seed=42):
    obs_diff, ci_low, ci_high, p_val, diffs, k1, k2, count_le, count_ge = \
        bootstrap_kappa_diff(cm_model1, cm_model2, n_boot=n_boot, seed=seed)

    p_str = format_p(p_val, n_boot)

    print("=" * 56)
    print(f"  {label}")
    print(f"  Bootstrap CI on Cohen's κ Difference ({n_boot:,} resamples)")
    print("=" * 56)
    print(f"  κ  (Base model)             = {k1:.4f}")
    print(f"  κ  (Proposed model)         = {k2:.4f}")
    print(f"  Observed Δκ                 = {obs_diff:+.4f}  (proposed − base)")
    print(f"  95% Bootstrap CI            = [{ci_low:.4f},  {ci_high:.4f}]")
    print(f"  Bootstrap p-value           = {p_str}")
    print(f"  (Resamples with Δκ ≤ 0 : {count_le} / {n_boot})")
    print()

    if ci_low > 0:
        print("  → CI entirely above 0: Proposed model SIGNIFICANTLY BETTER")
    elif ci_high < 0:
        print("  → CI entirely below 0: Base model SIGNIFICANTLY BETTER")
    else:
        print("  → CI includes 0: NO significant difference (p ≥ 0.05)")

    print()
    print(f"  Bootstrap distribution of Δκ:")
    print(f"    Mean  = {np.mean(diffs):+.4f}")
    print(f"    Std   = {np.std(diffs):.4f}")
    print(f"    Min   = {np.min(diffs):+.4f}")
    print(f"    Max   = {np.max(diffs):+.4f}")
    print()

# -----------------------------------------------
# Aggregated confusion matrices
# -----------------------------------------------

# IARC TriStream
cm1_r152_iarc = np.array([[174, 26, 26],
                       [ 12, 43,  8],
                       [ 13,  1, 73]])

cm2_r152_iarc = np.array([[182, 19, 25],
                       [ 12, 49,  2],
                       [ 16,  1, 70]])

# aNNoCERV TriStream
# cm1_r152_ann = np.array([[225, 46, 40],
#                        [ 23, 95,  6],
#                        [ 20,  3, 73]])

# cm2_r152_ann = np.array([[238, 40, 33],
#                        [ 25, 95,  4],
#                        [ 18,  2, 76]])

# -----------------------------------------------
# Run analyses
# -----------------------------------------------
run_analysis("IARC R152 vs BF-R152",    cm1_r152_iarc, cm2_r152_iarc)
#run_analysis("aNNoCERV TriStream", cm1_r152_ann,  cm2_r152_ann)

In [ ]:
#ResNet152
import numpy as np

# -----------------------------------------------
# Cohen's kappa
# -----------------------------------------------
def cohens_kappa(cm):
    N = np.sum(cm)
    Po = np.trace(cm) / N
    row = np.sum(cm, axis=1)
    col = np.sum(cm, axis=0)
    Pe = np.sum(row * col) / (N * N)
    kappa = (Po - Pe) / (1 - Pe)
    var_k = (Po * (1 - Po)) / (N * (1 - Pe)**2)
    return kappa, var_k

# -----------------------------------------------
# Bootstrap CI on kappa difference
# -----------------------------------------------
def bootstrap_kappa_diff(cm1, cm2, n_boot=100, seed=42):
    rng = np.random.default_rng(seed)
    N = int(cm1.sum())
    n_classes = cm1.shape[0]

    # Expand CM into sample-level (true_label, pred_label) pairs
    def cm_to_samples(cm):
        samples = []
        for i in range(n_classes):
            for j in range(n_classes):
                samples.extend([(i, j)] * cm[i, j])
        return np.array(samples)   # shape (N, 2)

    s1 = cm_to_samples(cm1)   # model1 samples
    s2 = cm_to_samples(cm2)   # model2 samples

    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, N, size=N)   # resample WITH replacement

        # Rebuild CMs from bootstrap indices
        boot_cm1 = np.zeros((n_classes, n_classes), dtype=int)
        boot_cm2 = np.zeros((n_classes, n_classes), dtype=int)
        for true_l, pred_l in s1[idx]:
            boot_cm1[true_l, pred_l] += 1
        for true_l, pred_l in s2[idx]:
            boot_cm2[true_l, pred_l] += 1

        k1, _ = cohens_kappa(boot_cm1)
        k2, _ = cohens_kappa(boot_cm2)
        diffs.append(k2 - k1)   # proposed - base

    diffs = np.array(diffs)

    # Observed difference
    k1_obs, _ = cohens_kappa(cm1)
    k2_obs, _ = cohens_kappa(cm2)
    obs_diff = k2_obs - k1_obs

    # Percentile CI
    ci_low  = np.percentile(diffs, 2.5)
    ci_high = np.percentile(diffs, 97.5)

    # Continuity-corrected two-sided bootstrap p-value.
    # Adding +1 to numerator and denominator avoids an exact p=0 when all
    # resamples fall on one side — a standard Monte Carlo correction.
    count_le = np.sum(diffs <= 0)
    count_ge = np.sum(diffs >= 0)
    p_val = 2 * min(
        (count_le + 1) / (n_boot + 1),
        (count_ge + 1) / (n_boot + 1)
    )

    return obs_diff, ci_low, ci_high, p_val, diffs, k1_obs, k2_obs, count_le, count_ge

# -----------------------------------------------
# Helper: format p-value honestly
# -----------------------------------------------
def format_p(p_val, n_boot):
    """
    When all bootstrap resamples fall on one side, the continuity-corrected
    p equals 2/(n_boot+1).  Report this as a bound rather than a spurious
    exact value.
    """
    resolution = 2 / (n_boot + 1)
    if p_val <= resolution:
        return f"< {resolution:.4f}  (below bootstrap resolution)"
    elif p_val < 0.001:
        return f"{p_val:.2e}"
    else:
        return f"{p_val:.4f}"

# -----------------------------------------------
# Pretty-print results for one experiment
# -----------------------------------------------
def run_analysis(label, cm_model1, cm_model2, n_boot=200, seed=42):
    obs_diff, ci_low, ci_high, p_val, diffs, k1, k2, count_le, count_ge = \
        bootstrap_kappa_diff(cm_model1, cm_model2, n_boot=n_boot, seed=seed)

    p_str = format_p(p_val, n_boot)

    print("=" * 56)
    print(f"  {label}")
    print(f"  Bootstrap CI on Cohen's κ Difference ({n_boot:,} resamples)")
    print("=" * 56)
    print(f"  κ  (Base model)             = {k1:.4f}")
    print(f"  κ  (Proposed model)         = {k2:.4f}")
    print(f"  Observed Δκ                 = {obs_diff:+.4f}  (proposed − base)")
    print(f"  95% Bootstrap CI            = [{ci_low:.4f},  {ci_high:.4f}]")
    print(f"  Bootstrap p-value           = {p_str}")
    print(f"  (Resamples with Δκ ≤ 0 : {count_le} / {n_boot})")
    print()

    if ci_low > 0:
        print("  → CI entirely above 0: Proposed model SIGNIFICANTLY BETTER")
    elif ci_high < 0:
        print("  → CI entirely below 0: Base model SIGNIFICANTLY BETTER")
    else:
        print("  → CI includes 0: NO significant difference (p ≥ 0.05)")

    print()
    print(f"  Bootstrap distribution of Δκ:")
    print(f"    Mean  = {np.mean(diffs):+.4f}")
    print(f"    Std   = {np.std(diffs):.4f}")
    print(f"    Min   = {np.min(diffs):+.4f}")
    print(f"    Max   = {np.max(diffs):+.4f}")
    print()

# -----------------------------------------------
# Aggregated confusion matrices
# -----------------------------------------------

# # IARC TriStream
# cm1_r152_iarc = np.array([[174, 26, 26],
#                        [ 12, 43,  8],
#                        [ 13,  1, 73]])

# cm2_r152_iarc = np.array([[182, 19, 25],
#                        [ 12, 49,  2],
#                        [ 16,  1, 70]])

# aNNoCERV TriStream
cm1_r152_ann = np.array([[225, 46, 40],
                       [ 23, 95,  6],
                       [ 20,  3, 73]])

cm2_r152_ann = np.array([[238, 40, 33],
                       [ 25, 95,  4],
                       [ 18,  2, 76]])

# -----------------------------------------------
# Run analyses
# -----------------------------------------------
#run_analysis("IARC TriStream",    cm1_r152_iarc, cm2_r152_iarc)
run_analysis("ANNoCERV R152 vs BF-R152", cm1_r152_ann,  cm2_r152_ann)

In [ ]:
#Tristream
import numpy as np

# -----------------------------------------------
# Cohen's kappa
# -----------------------------------------------
def cohens_kappa(cm):
    N = np.sum(cm)
    Po = np.trace(cm) / N
    row = np.sum(cm, axis=1)
    col = np.sum(cm, axis=0)
    Pe = np.sum(row * col) / (N * N)
    kappa = (Po - Pe) / (1 - Pe)
    var_k = (Po * (1 - Po)) / (N * (1 - Pe)**2)
    return kappa, var_k

# -----------------------------------------------
# Bootstrap CI on kappa difference
# -----------------------------------------------
def bootstrap_kappa_diff(cm1, cm2, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    N = int(cm1.sum())
    n_classes = cm1.shape[0]

    # Expand CM into sample-level (true_label, pred_label) pairs
    def cm_to_samples(cm):
        samples = []
        for i in range(n_classes):
            for j in range(n_classes):
                samples.extend([(i, j)] * cm[i, j])
        return np.array(samples)   # shape (N, 2)

    s1 = cm_to_samples(cm1)   # model1 samples
    s2 = cm_to_samples(cm2)   # model2 samples

    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, N, size=N)   # resample WITH replacement

        # Rebuild CMs from bootstrap indices
        boot_cm1 = np.zeros((n_classes, n_classes), dtype=int)
        boot_cm2 = np.zeros((n_classes, n_classes), dtype=int)
        for true_l, pred_l in s1[idx]:
            boot_cm1[true_l, pred_l] += 1
        for true_l, pred_l in s2[idx]:
            boot_cm2[true_l, pred_l] += 1

        k1, _ = cohens_kappa(boot_cm1)
        k2, _ = cohens_kappa(boot_cm2)
        diffs.append(k2 - k1)   # proposed - base

    diffs = np.array(diffs)

    # Observed difference
    k1_obs, _ = cohens_kappa(cm1)
    k2_obs, _ = cohens_kappa(cm2)
    obs_diff = k2_obs - k1_obs

    # Percentile CI
    ci_low  = np.percentile(diffs, 2.5)
    ci_high = np.percentile(diffs, 97.5)

    # Continuity-corrected two-sided bootstrap p-value.
    # Adding +1 to numerator and denominator avoids an exact p=0 when all
    # resamples fall on one side — a standard Monte Carlo correction.
    count_le = np.sum(diffs <= 0)
    count_ge = np.sum(diffs >= 0)
    p_val = 2 * min(
        (count_le + 1) / (n_boot + 1),
        (count_ge + 1) / (n_boot + 1)
    )

    return obs_diff, ci_low, ci_high, p_val, diffs, k1_obs, k2_obs, count_le, count_ge

# -----------------------------------------------
# Helper: format p-value honestly
# -----------------------------------------------
def format_p(p_val, n_boot):
    """
    When all bootstrap resamples fall on one side, the continuity-corrected
    p equals 2/(n_boot+1).  Report this as a bound rather than a spurious
    exact value.
    """
    resolution = 2 / (n_boot + 1)
    if p_val <= resolution:
        return f"< {resolution:.4f}  (below bootstrap resolution)"
    elif p_val < 0.001:
        return f"{p_val:.2e}"
    else:
        return f"{p_val:.4f}"

# -----------------------------------------------
# Pretty-print results for one experiment
# -----------------------------------------------
def run_analysis(label, cm_model1, cm_model2, n_boot=10000, seed=42):
    obs_diff, ci_low, ci_high, p_val, diffs, k1, k2, count_le, count_ge = \
        bootstrap_kappa_diff(cm_model1, cm_model2, n_boot=n_boot, seed=seed)

    p_str = format_p(p_val, n_boot)

    print("=" * 56)
    print(f"  {label}")
    print(f"  Bootstrap CI on Cohen's κ Difference ({n_boot:,} resamples)")
    print("=" * 56)
    print(f"  κ  (Base model)             = {k1:.4f}")
    print(f"  κ  (Proposed model)         = {k2:.4f}")
    print(f"  Observed Δκ                 = {obs_diff:+.4f}  (proposed − base)")
    print(f"  95% Bootstrap CI            = [{ci_low:.4f},  {ci_high:.4f}]")
    print(f"  Bootstrap p-value           = {p_str}")
    print(f"  (Resamples with Δκ ≤ 0 : {count_le} / {n_boot})")
    print()

    if ci_low > 0:
        print("  → CI entirely above 0: Proposed model SIGNIFICANTLY BETTER")
    elif ci_high < 0:
        print("  → CI entirely below 0: Base model SIGNIFICANTLY BETTER")
    else:
        print("  → CI includes 0: NO significant difference (p ≥ 0.05)")

    print()
    print(f"  Bootstrap distribution of Δκ:")
    print(f"    Mean  = {np.mean(diffs):+.4f}")
    print(f"    Std   = {np.std(diffs):.4f}")
    print(f"    Min   = {np.min(diffs):+.4f}")
    print(f"    Max   = {np.max(diffs):+.4f}")
    print()

# -----------------------------------------------
# Aggregated confusion matrices
# -----------------------------------------------

# IARC TriStream
cm1_tristream_iarc = np.array([[189, 16, 21],
                     [  9, 48,  6],
                     [ 11,  0, 76]])

cm2_tristream_iarc = np.array([[207, 17,  8],
                     [  8, 51,  4],
                     [  6,  0, 81]])

# aNNoCERV TriStream
cm1_tristream_ann = np.array([[243, 43, 25],
                    [ 14, 105,  5],
                    [ 17,   2, 77]])

cm2_tristream_ann = np.array([[262, 23, 26],
                    [ 16, 106,  2],
                    [ 14,   2, 80]])

# -----------------------------------------------
# Run analyses
# -----------------------------------------------
run_analysis("IARC TriStream",    cm1_tristream_iarc, cm2_tristream_iarc)
run_analysis("aNNoCERV TriStream", cm1_tristream_ann,  cm2_tristream_ann)

In [ ]:
# Tristream VS RESNET152
import numpy as np

# -----------------------------------------------
# Cohen's kappa
# -----------------------------------------------
def cohens_kappa(cm):
    N = np.sum(cm)
    Po = np.trace(cm) / N
    row = np.sum(cm, axis=1)
    col = np.sum(cm, axis=0)
    Pe = np.sum(row * col) / (N * N)
    kappa = (Po - Pe) / (1 - Pe)
    var_k = (Po * (1 - Po)) / (N * (1 - Pe)**2)
    return kappa, var_k

# -----------------------------------------------
# Bootstrap CI on kappa difference (FIXED)
# -----------------------------------------------
def bootstrap_kappa_diff(cm1, cm2, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    N1 = int(cm1.sum())   # independent sizes for each model
    N2 = int(cm2.sum())
    n_classes = cm1.shape[0]

    # Expand CM into sample-level (true_label, pred_label) pairs
    def cm_to_samples(cm):
        samples = []
        for i in range(n_classes):
            for j in range(n_classes):
                samples.extend([(i, j)] * cm[i, j])
        return np.array(samples)   # shape (N, 2)

    s1 = cm_to_samples(cm1)   # model1 samples
    s2 = cm_to_samples(cm2)   # model2 samples

    diffs = []
    for _ in range(n_boot):
        idx1 = rng.integers(0, N1, size=N1)   # independent resample for model 1
        idx2 = rng.integers(0, N2, size=N2)   # independent resample for model 2

        # Rebuild CMs from bootstrap indices
        boot_cm1 = np.zeros((n_classes, n_classes), dtype=int)
        boot_cm2 = np.zeros((n_classes, n_classes), dtype=int)
        for true_l, pred_l in s1[idx1]:
            boot_cm1[true_l, pred_l] += 1
        for true_l, pred_l in s2[idx2]:
            boot_cm2[true_l, pred_l] += 1

        k1, _ = cohens_kappa(boot_cm1)
        k2, _ = cohens_kappa(boot_cm2)
        diffs.append(k2 - k1)   # proposed - base

    diffs = np.array(diffs)

    # Observed difference
    k1_obs, _ = cohens_kappa(cm1)
    k2_obs, _ = cohens_kappa(cm2)
    obs_diff = k2_obs - k1_obs

    # Percentile CI
    ci_low  = np.percentile(diffs, 2.5)
    ci_high = np.percentile(diffs, 97.5)

    # Continuity-corrected two-sided bootstrap p-value.
    # Adding +1 to numerator and denominator avoids an exact p=0 when all
    # resamples fall on one side — a standard Monte Carlo correction.
    count_le = np.sum(diffs <= 0)
    count_ge = np.sum(diffs >= 0)
    p_val = 2 * min(
        (count_le + 1) / (n_boot + 1),
        (count_ge + 1) / (n_boot + 1)
    )

    return obs_diff, ci_low, ci_high, p_val, diffs, k1_obs, k2_obs, count_le, count_ge

# -----------------------------------------------
# Helper: format p-value honestly
# -----------------------------------------------
def format_p(p_val, n_boot):
    """
    When all bootstrap resamples fall on one side, the continuity-corrected
    p equals 2/(n_boot+1).  Report this as a bound rather than a spurious
    exact value.
    """
    resolution = 2 / (n_boot + 1)
    if p_val <= resolution:
        return f"< {resolution:.4f}  (below bootstrap resolution)"
    elif p_val < 0.001:
        return f"{p_val:.2e}"
    else:
        return f"{p_val:.4f}"

# -----------------------------------------------
# Pretty-print results for one experiment
# -----------------------------------------------
def run_analysis(label, cm_model1, cm_model2, n_boot=10000, seed=42):
    obs_diff, ci_low, ci_high, p_val, diffs, k1, k2, count_le, count_ge = \
        bootstrap_kappa_diff(cm_model1, cm_model2, n_boot=n_boot, seed=seed)

    p_str = format_p(p_val, n_boot)

    print("=" * 56)
    print(f"  {label}")
    print(f"  Bootstrap CI on Cohen's κ Difference ({n_boot:,} resamples)")
    print("=" * 56)
    print(f"  κ  (Base model)             = {k1:.4f}")
    print(f"  κ  (Proposed model)         = {k2:.4f}")
    print(f"  Observed Δκ                 = {obs_diff:+.4f}  (proposed − base)")
    print(f"  95% Bootstrap CI            = [{ci_low:.4f},  {ci_high:.4f}]")
    print(f"  Bootstrap p-value           = {p_str}")
    print(f"  (Resamples with Δκ ≤ 0 : {count_le} / {n_boot})")
    print()

    if ci_low > 0:
        print("  → CI entirely above 0: Proposed model SIGNIFICANTLY BETTER")
    elif ci_high < 0:
        print("  → CI entirely below 0: Base model SIGNIFICANTLY BETTER")
    else:
        print("  → CI includes 0: NO significant difference (p ≥ 0.05)")

    print()
    print(f"  Bootstrap distribution of Δκ:")
    print(f"    Mean  = {np.mean(diffs):+.4f}")
    print(f"    Std   = {np.std(diffs):.4f}")
    print(f"    Min   = {np.min(diffs):+.4f}")
    print(f"    Max   = {np.max(diffs):+.4f}")
    print()

# -----------------------------------------------
# Aggregated confusion matrices
# -----------------------------------------------

# IARC — ResNet152 (base) vs TriStream (proposed)
cm1_r152_iarc = np.array([[174, 26, 26],
                           [ 12, 43,  8],
                           [ 13,  1, 73]])

cm2_tristream_iarc = np.array([[207, 17,  2],
                                [  8, 51,  4],
                                [  6,  0, 81]])

# aNNoCERV — ResNet152 (base) vs TriStream (proposed)
cm1_r152_ann = np.array([[225, 46, 40],
                          [ 23, 95,  6],
                          [ 20,  3, 73]])

cm2_tristream_ann = np.array([[262, 23, 26],
                               [ 16, 106,  2],
                               [ 14,   2, 80]])

# -----------------------------------------------
# Run analyses
# -----------------------------------------------
run_analysis("IARC R152 Vs TriStream",     cm1_r152_iarc, cm2_tristream_iarc)
run_analysis("aNNoCERV R152 Vs TriStream", cm1_r152_ann,  cm2_tristream_ann)